# VLA 연구 · 11회차 실습
## MuJoCo 씬을 LeRobot-VLA 스타일 action chunk로 제어하기

| 항목 | 내용 |
|------|------|
| 오늘 목표 | MuJoCo 정리 씬에서 `observation → task → action chunk → control` 흐름을 직접 실행한다. |
| 프로젝트 연결 | 10회차에서 만든 과정 라벨이 실제 제어 가능한 라벨인지 검수한다. |
| 핵심 아이디어 | 안정적인 scripted teacher를 `MiniVLAController.select_action()` 인터페이스로 감싸 VLA 입출력 구조를 체험한다. |
| 최종 산출물 | rollout 이미지/GIF, trajectory JSON, 제어 가능성 기준 라벨 검수표 |

이번 실습은 pretrained VLA 성능 확인이 아니라, “라벨이 실제 제어 단위로 적절한가?”를 확인하는 수업입니다. 참고 흐름은 `lerobot-mujoco-tutorial`의 demonstration data, language-conditioned environment, SmolVLA deploy 구조입니다.

참고: https://github.com/jeongeun980906/lerobot-mujoco-tutorial


## SECTION 0  |  실행 환경과 작업 폴더 준비

Colab에서는 첫 설치 뒤 런타임 재시작이 필요할 수 있습니다. 기본 경로는 Colab과 로컬 모두에서 동작하도록 작성했습니다.


In [ ]:
!pip install lerobot

In [ ]:
import sys
import pkgutil
from importlib import metadata

import lerobot

print("Python:", sys.version)
print("Python executable:", sys.executable)
print("LeRobot version:", metadata.version("lerobot"))
print("LeRobot loaded from:", lerobot.__file__)

if hasattr(lerobot, "__path__"):
    subpackages = sorted(m.name for m in pkgutil.iter_modules(lerobot.__path__))
    print("LeRobot subpackages:", subpackages)

In [ ]:
# 0-1. 패키지 설치

import importlib.util
import os
import subprocess
import sys

os.environ.setdefault('MUJOCO_GL', 'egl')

PACKAGES = ['mujoco', 'imageio', 'mediapy', 'pandas']
missing = [pkg for pkg in PACKAGES if importlib.util.find_spec(pkg) is None]
if missing:
    print('설치가 필요한 패키지:', missing)
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q'] + missing)
else:
    print('필수 패키지가 이미 설치되어 있습니다.')
print('MUJOCO_GL =', os.environ.get('MUJOCO_GL'))


In [ ]:
# 0-2. 기본 import, 작업 경로, Drive 연결

from pathlib import Path
import json
import os
import platform
import shutil
import sys

import imageio.v2 as imageio
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from PIL import Image
from IPython.display import display

try:
    import mujoco
    MUJOCO_AVAILABLE = True
    print('mujoco:', mujoco.__version__)
except Exception as e:
    MUJOCO_AVAILABLE = False
    print('MuJoCo import 실패:', repr(e))

IN_COLAB = 'google.colab' in sys.modules or os.path.exists('/content')
WORK_DIR = Path('/content/session11_mujoco_vla') if IN_COLAB else Path.cwd() / 'session11_mujoco_vla'
DATA_DIR = WORK_DIR / 'data'
XML_DIR = DATA_DIR / 'xml'
IMAGE_DIR = DATA_DIR / 'images'
GIF_DIR = DATA_DIR / 'gifs'
TRAJ_DIR = DATA_DIR / 'trajectories'
RESULT_DIR = DATA_DIR / 'results'
for path in [WORK_DIR, DATA_DIR, XML_DIR, IMAGE_DIR, GIF_DIR, TRAJ_DIR, RESULT_DIR]:
    path.mkdir(parents=True, exist_ok=True)

TEAM_ROOT = None
if IN_COLAB:
    try:
        from google.colab import drive
        drive.mount('/content/drive')
        TEAM_ROOT = Path('/content/drive/MyDrive/VLA_Team_2026')
        print('TEAM_ROOT:', TEAM_ROOT)
    except Exception as e:
        print('Drive 마운트 생략:', repr(e))

print('Python:', sys.version.split()[0])
print('Platform:', platform.platform())
print('WORK_DIR:', WORK_DIR)


## SECTION 1  |  10회차 정리 시나리오와 라벨 복습

10회차의 다섯 `sort_story` 상태와 과정 라벨을 그대로 사용합니다. 오늘은 각 라벨이 실제 제어 primitive로 실행 가능한지 검수합니다.


In [ ]:
# 1-1. 10회차 시나리오 재정의

TABLE_TOP_Z = 0.425

OBJECT_SPECS = {
    'target_red_box': {'geom_type': 'box', 'size': (0.045, 0.045, 0.040), 'rgba': '0.95 0.10 0.10 1.0', 'role': 'target_to_left_zone'},
    'blue_blocker': {'geom_type': 'box', 'size': (0.060, 0.035, 0.035), 'rgba': '0.10 0.25 0.95 1.0', 'role': 'obstacle_to_right_side'},
    'yellow_cylinder': {'geom_type': 'cylinder', 'size': (0.035, 0.040), 'rgba': '0.95 0.75 0.10 1.0', 'role': 'object_to_back_zone'},
    'gray_box': {'geom_type': 'box', 'size': (0.040, 0.040, 0.030), 'rgba': '0.45 0.45 0.45 1.0', 'role': 'object_to_right_zone'},
}

ZONES = {
    'left_zone': {'pos': (-0.25, 0.12, TABLE_TOP_Z + 0.004), 'size': (0.105, 0.080, 0.004), 'rgba': '0.20 0.75 0.35 0.35'},
    'right_zone': {'pos': (0.25, 0.02, TABLE_TOP_Z + 0.004), 'size': (0.105, 0.080, 0.004), 'rgba': '0.35 0.55 0.95 0.35'},
    'back_zone': {'pos': (0.00, 0.23, TABLE_TOP_Z + 0.004), 'size': (0.105, 0.070, 0.004), 'rgba': '0.95 0.70 0.20 0.35'},
}

SCENARIO_STATES = [
    {'scene_id': 'sort_story_00_needs_sorting', 'scene_status': 'needs_sorting', 'state_caption': '정리가 필요한 초기 상태: 파란 블록이 빨간 상자 앞을 막고 여러 물체가 흩어져 있다.', 'draft_next_process': '파란 장애물을 오른쪽으로 밀어 빨간 상자로 가는 작업 경로를 만든다.', 'positions': {'target_red_box': (0.00, 0.06), 'blue_blocker': (0.00, -0.08), 'yellow_cylinder': (-0.10, 0.13), 'gray_box': (0.12, 0.02)}},
    {'scene_id': 'sort_story_01_after_push_right', 'scene_status': 'in_progress', 'state_caption': '파란 블록을 오른쪽으로 밀어 목표 상자 앞 경로가 열렸다.', 'draft_next_process': '빨간 목표 상자를 들어서 왼쪽 보관 영역으로 옮긴다.', 'positions': {'target_red_box': (0.00, 0.06), 'blue_blocker': (0.25, -0.08), 'yellow_cylinder': (-0.10, 0.13), 'gray_box': (0.12, 0.02)}},
    {'scene_id': 'sort_story_02_after_lift_left', 'scene_status': 'in_progress', 'state_caption': '빨간 목표 상자가 왼쪽 보관 영역에 놓였지만 아직 노란 원통과 회색 블록이 남아 있다.', 'draft_next_process': '노란 원통을 들어서 뒤쪽 정리 영역으로 옮긴다.', 'positions': {'target_red_box': (-0.25, 0.12), 'blue_blocker': (0.25, -0.08), 'yellow_cylinder': (-0.10, 0.13), 'gray_box': (0.12, 0.02)}},
    {'scene_id': 'sort_story_03_after_yellow_back', 'scene_status': 'in_progress', 'state_caption': '노란 원통이 뒤쪽 정리 영역에 놓였고 회색 블록 하나가 아직 정리되지 않았다.', 'draft_next_process': '회색 블록을 들어서 오른쪽 정리 영역으로 옮긴다.', 'positions': {'target_red_box': (-0.25, 0.12), 'blue_blocker': (0.25, -0.08), 'yellow_cylinder': (0.00, 0.23), 'gray_box': (0.12, 0.02)}},
    {'scene_id': 'sort_story_04_sorted_done', 'scene_status': 'sorted', 'state_caption': '정리가 완료된 상태: 각 물체가 지정된 영역에 놓여 있다.', 'draft_next_process': '정리 완료 상태를 확인하고 더 이상 움직이지 않는다.', 'positions': {'target_red_box': (-0.25, 0.12), 'blue_blocker': (0.25, -0.08), 'yellow_cylinder': (0.00, 0.23), 'gray_box': (0.25, 0.02)}},
]

PROCESS_TAXONOMY = {
    'slide_obstacle_right': {'process_type': 'planar_push', 'short_ko': '오른쪽으로 밀기', 'definition': '장애물을 집지 않고 테이블 위에서 오른쪽으로 밀어 경로를 확보한다.', 'legacy_action_optional': 'push'},
    'lift_target_to_left_zone': {'process_type': 'pick_and_place', 'short_ko': '들어서 왼쪽으로 옮기기', 'definition': '목표 물체를 들어 왼쪽 보관 영역에 놓는다.', 'legacy_action_optional': 'pick'},
    'lift_object_to_back_zone': {'process_type': 'pick_and_place', 'short_ko': '들어서 뒤쪽으로 옮기기', 'definition': '남은 물체를 들어 뒤쪽 정리 영역에 놓는다.', 'legacy_action_optional': 'reorder'},
    'lift_object_to_right_zone': {'process_type': 'pick_and_place', 'short_ko': '들어서 오른쪽으로 옮기기', 'definition': '남은 물체를 들어 오른쪽 정리 영역에 놓는다.', 'legacy_action_optional': 'reorder'},
    'confirm_sorted_stop': {'process_type': 'terminal_check', 'short_ko': '정리 완료 확인', 'definition': '정리가 끝났는지 확인하고 추가 조작 없이 멈춘다.', 'legacy_action_optional': 'ignore'},
}

PROCESS_ASSIGNMENTS = {
    'sort_story_00_needs_sorting': 'slide_obstacle_right',
    'sort_story_01_after_push_right': 'lift_target_to_left_zone',
    'sort_story_02_after_lift_left': 'lift_object_to_back_zone',
    'sort_story_03_after_yellow_back': 'lift_object_to_right_zone',
    'sort_story_04_sorted_done': 'confirm_sorted_stop',
}

scenario_df = pd.DataFrame([
    {'scene_id': s['scene_id'], 'status': s['scene_status'], 'process_label': PROCESS_ASSIGNMENTS[s['scene_id']], 'process_type': PROCESS_TAXONOMY[PROCESS_ASSIGNMENTS[s['scene_id']]]['process_type'], 'short_ko': PROCESS_TAXONOMY[PROCESS_ASSIGNMENTS[s['scene_id']]]['short_ko'], 'draft_next_process': s['draft_next_process']}
    for s in SCENARIO_STATES
])
display(scenario_df)


## SECTION 2  |  MuJoCo XML 생성과 렌더링

오늘은 외부 robot asset 없이 실행되는 LeRobot-compatible OMY-style 7차원 tabletop arm을 사용합니다. 실제 LeRobot/SmolVLA 실험으로 확장할 때도 `observation.state`, `observation.image`, `task`, `action`의 연결 구조는 같습니다.


여기서 “LeRobot 로봇”은 LeRobot 프레임워크가 기대하는 `state/action/task/image` 인터페이스에 맞춘 교육용 OMY-style arm을 뜻합니다. 실제 튜토리얼의 OMY asset을 그대로 포함하지 않고, Colab에서 안정적으로 실행되도록 같은 7차원 action 구조만 가볍게 재현합니다.


In [ ]:
# 2-1. MuJoCo XML 생성 함수

def object_z(geom_type, size):
    if geom_type == 'box':
        return TABLE_TOP_Z + size[2]
    if geom_type == 'cylinder':
        return TABLE_TOP_Z + size[1]
    return TABLE_TOP_Z + size[-1]


def object_body_xml(name, xy, spec):
    z = object_z(spec['geom_type'], spec['size'])
    size_str = ' '.join(f'{v:.4f}' for v in spec['size'])
    return f'''
    <body name="{name}" pos="{xy[0]:.4f} {xy[1]:.4f} {z:.4f}">
      <freejoint name="{name}_freejoint"/>
      <geom name="{name}_geom" type="{spec['geom_type']}" size="{size_str}" rgba="{spec['rgba']}" mass="0.20" friction="0.8 0.02 0.001"/>
    </body>'''


def zone_geom_xml(name, zone):
    pos = zone['pos']
    size = zone['size']
    return f'''<geom name="{name}" type="box" pos="{pos[0]:.4f} {pos[1]:.4f} {pos[2]:.4f}" size="{size[0]:.4f} {size[1]:.4f} {size[2]:.4f}" rgba="{zone['rgba']}" contype="0" conaffinity="0"/>'''


# LeRobot tutorial의 OMY 환경처럼 6 joint + 1 gripper action을 받는 교육용 arm입니다.
# 실제 asset을 다운로드하지 않아도 수업 중 action/state 흐름을 확인할 수 있게 단순화했습니다.
def lerobot_omy_arm_xml():
    return '''
    <body name="lerobot_omy_base" pos="0 -0.42 0.425">
      <geom name="base_geom" type="cylinder" size="0.055 0.040" rgba="0.15 0.15 0.18 1"/>
      <body name="link1" pos="0 0 0.055">
        <joint name="joint1" type="hinge" axis="0 0 1" range="-180 180" damping="1.0"/>
        <geom type="capsule" fromto="0 0 0 0 0 0.15" size="0.018" rgba="0.25 0.25 0.28 1"/>
        <body name="link2" pos="0 0 0.15">
          <joint name="joint2" type="hinge" axis="1 0 0" range="-90 90" damping="1.0"/>
          <geom type="capsule" fromto="0 0 0 0 0.18 0.04" size="0.016" rgba="0.35 0.35 0.38 1"/>
          <body name="link3" pos="0 0.18 0.04">
            <joint name="joint3" type="hinge" axis="1 0 0" range="-120 120" damping="1.0"/>
            <geom type="capsule" fromto="0 0 0 0 0.16 -0.03" size="0.014" rgba="0.45 0.45 0.48 1"/>
            <body name="wrist" pos="0 0.16 -0.03">
              <joint name="joint4" type="hinge" axis="0 1 0" range="-120 120" damping="0.8"/>
              <joint name="joint5" type="hinge" axis="1 0 0" range="-120 120" damping="0.8"/>
              <joint name="joint6" type="hinge" axis="0 0 1" range="-180 180" damping="0.8"/>
              <geom name="wrist_geom" type="sphere" size="0.025" rgba="0.10 0.10 0.12 1"/>
              <body name="gripper_left" pos="-0.025 0.035 0">
                <joint name="gripper" type="slide" axis="1 0 0" range="0 0.035" damping="0.5"/>
                <geom type="box" size="0.008 0.035 0.010" rgba="0.08 0.08 0.09 1"/>
              </body>
              <body name="gripper_right" pos="0.025 0.035 0">
                <geom type="box" size="0.008 0.035 0.010" rgba="0.08 0.08 0.09 1"/>
              </body>
            </body>
          </body>
        </body>
      </body>
    </body>'''


def make_sorting_scene_xml(state):
    object_xml = '\n'.join(object_body_xml(name, xy, OBJECT_SPECS[name]) for name, xy in state['positions'].items())
    zone_xml = '\n'.join(zone_geom_xml(name, zone) for name, zone in ZONES.items())
    return f'''<mujoco model="session11_lerobot_vla_sorting_scene">
  <compiler angle="degree" coordinate="local"/>
  <option timestep="0.02" gravity="0 0 -9.81"/>
  <visual><global offwidth="640" offheight="480"/></visual>
  <asset>
    <texture name="grid" type="2d" builtin="checker" width="512" height="512" rgb1="0.86 0.86 0.82" rgb2="0.78 0.78 0.74"/>
    <material name="gridmat" texture="grid" texrepeat="4 4" reflectance="0.1"/>
  </asset>
  <worldbody>
    <light name="key" pos="0 -1.2 2.5" diffuse="0.9 0.9 0.9" specular="0.2 0.2 0.2"/>
    <camera name="vlm_camera" pos="0.80 -0.90 1.15" xyaxes="1 0.45 0 -0.30 0.65 0.70"/>
    <geom name="floor" type="plane" pos="0 0 0" size="1.5 1.5 0.02" material="gridmat"/>
    <geom name="table" type="box" pos="0 0 {TABLE_TOP_Z / 2:.4f}" size="0.48 0.38 {TABLE_TOP_Z / 2:.4f}" rgba="0.55 0.45 0.35 1"/>
    {zone_xml}
    {object_xml}
    {lerobot_omy_arm_xml()}
  </worldbody>
  <actuator>
    <position name="a1" joint="joint1" kp="20" ctrlrange="-3.14 3.14"/>
    <position name="a2" joint="joint2" kp="20" ctrlrange="-1.57 1.57"/>
    <position name="a3" joint="joint3" kp="20" ctrlrange="-2.09 2.09"/>
    <position name="a4" joint="joint4" kp="12" ctrlrange="-2.09 2.09"/>
    <position name="a5" joint="joint5" kp="12" ctrlrange="-2.09 2.09"/>
    <position name="a6" joint="joint6" kp="12" ctrlrange="-3.14 3.14"/>
    <position name="gripper_ctrl" joint="gripper" kp="8" ctrlrange="0 0.035"/>
  </actuator>
</mujoco>'''


In [ ]:
# 2-2. 렌더링과 object pose 헬퍼

def load_model_from_xml(xml_text):
    if not MUJOCO_AVAILABLE:
        raise RuntimeError('MuJoCo가 import되지 않았습니다.')
    model = mujoco.MjModel.from_xml_string(xml_text)
    data = mujoco.MjData(model)
    mujoco.mj_forward(model, data)
    return model, data


def render_rgb(model, data, camera='vlm_camera', width=640, height=480):
    renderer = mujoco.Renderer(model, height=height, width=width)
    renderer.update_scene(data, camera=camera)
    image = renderer.render().copy()
    renderer.close()
    return image


def freejoint_qpos_adr(model, joint_name):
    jid = mujoco.mj_name2id(model, mujoco.mjtObj.mjOBJ_JOINT, joint_name)
    if jid < 0:
        raise KeyError(f'joint not found: {joint_name}')
    return int(model.jnt_qposadr[jid])


def get_object_xyz(model, data, object_name):
    adr = freejoint_qpos_adr(model, f'{object_name}_freejoint')
    return data.qpos[adr:adr + 3].copy()


def set_object_xyz(model, data, object_name, xyz):
    adr = freejoint_qpos_adr(model, f'{object_name}_freejoint')
    data.qpos[adr:adr + 3] = np.asarray(xyz, dtype=np.float64)
    data.qpos[adr + 3:adr + 7] = np.array([1.0, 0.0, 0.0, 0.0])
    mujoco.mj_forward(model, data)


def get_all_object_positions(model, data):
    return {name: get_object_xyz(model, data, name).round(4).tolist() for name in OBJECT_SPECS}


def zone_center_xy(zone_name):
    return np.array(ZONES[zone_name]['pos'][:2], dtype=float)


def save_image(image, path):
    Image.fromarray(image).save(path)
    return str(path)


In [ ]:
# 2-3. 다섯 기본 씬 렌더링

scene_records = []
if MUJOCO_AVAILABLE:
    for state in SCENARIO_STATES:
        xml_text = make_sorting_scene_xml(state)
        xml_path = XML_DIR / f'{state["scene_id"]}.xml'
        xml_path.write_text(xml_text, encoding='utf-8')
        model, data = load_model_from_xml(xml_text)
        image = render_rgb(model, data)
        image_path = IMAGE_DIR / f'{state["scene_id"]}_initial.png'
        save_image(image, image_path)
        process_label = PROCESS_ASSIGNMENTS[state['scene_id']]
        scene_records.append({**state, 'process_label': process_label, 'process_type': PROCESS_TAXONOMY[process_label]['process_type'], 'xml_path': str(xml_path), 'initial_image_path': str(image_path), 'initial_object_positions': get_all_object_positions(model, data)})
        print('saved:', image_path.name, '|', process_label)

    fig, axes = plt.subplots(1, len(scene_records), figsize=(18, 4))
    for ax, record in zip(axes, scene_records):
        ax.imshow(Image.open(record['initial_image_path']))
        ax.set_title(record['scene_id'].replace('sort_story_', '') + '\n' + record['process_label'], fontsize=9)
        ax.axis('off')
    plt.tight_layout()
    plt.show()
else:
    print('MuJoCo를 사용할 수 없어 렌더링을 건너뜁니다.')


## SECTION 3  |  제어 관점에서 보는 VLA 입출력

VLA 입력은 이미지, 로봇 상태, 자연어 task를 함께 갖습니다. 출력은 한 timestep의 action이 아니라 여러 timestep의 action 묶음인 `action_chunk`일 수 있습니다.

오늘의 입력 `data`: `observation.image`, `observation.state`, `task`, `scene_id`, `process_label`  
오늘의 출력 `action_chunk`: shape `(T, 7)`인 NumPy 배열


In [ ]:
# 3-1. observation dict 만들기

ACTION_DIM = 7
STATE_DIM = 7

PROCESS_TASKS = {
    'slide_obstacle_right': 'Push the blue blocker to the right to open the path to the red box.',
    'lift_target_to_left_zone': 'Pick up the red target box and place it in the left green zone.',
    'lift_object_to_back_zone': 'Pick up the yellow cylinder and place it in the back orange zone.',
    'lift_object_to_right_zone': 'Pick up the gray box and place it in the right blue zone.',
    'confirm_sorted_stop': 'Check that the scene is sorted and stop moving.',
}


def robot_state_vector(model, data):
    state = np.zeros(STATE_DIM, dtype=np.float32)
    take = min(STATE_DIM, data.qpos.shape[0])
    state[:take] = data.qpos[:take]
    return state


def make_vla_input(record, model, data, image):
    label = record['process_label']
    return {'observation.image': image, 'observation.state': robot_state_vector(model, data), 'task': PROCESS_TASKS[label], 'scene_id': record['scene_id'], 'process_label': label}

if scene_records:
    sample_record = scene_records[0]
    sample_model, sample_data = load_model_from_xml(Path(sample_record['xml_path']).read_text(encoding='utf-8'))
    sample_image = render_rgb(sample_model, sample_data)
    sample_input = make_vla_input(sample_record, sample_model, sample_data, sample_image)
    print('VLA input keys:', list(sample_input.keys()))
    print('state shape:', sample_input['observation.state'].shape)
    print('image shape:', sample_input['observation.image'].shape)
    print('task:', sample_input['task'])


## SECTION 4  |  Scripted teacher policy로 제어 primitive 만들기

실제 물리 grasp는 접촉, 마찰, gripper 형상, inverse kinematics가 모두 맞아야 안정적입니다. 오늘은 라벨 검수를 위해 로봇 관절 접근 경로를 실행하면서 대상 물체 위치를 목표 위치로 부드럽게 갱신하는 교육용 primitive를 사용합니다.

이 방식은 “진짜 grasp 성공”을 증명하지 않습니다. 대신 라벨이 제어 단위로 너무 큰지, 너무 모호한지 확인하는 데 집중합니다.


In [ ]:
# 4-1. action chunk 생성 헬퍼

HOME_OPEN = np.array([0.00, -0.35, 0.75, -0.30, 0.00, 0.00, 0.030], dtype=np.float32)
APPROACH_LOW = np.array([0.00, -0.65, 0.95, -0.55, 0.00, 0.00, 0.030], dtype=np.float32)
GRASP_LOW = np.array([0.00, -0.70, 1.05, -0.60, 0.00, 0.00, 0.004], dtype=np.float32)
LIFT_POSE = np.array([0.00, -0.30, 0.65, -0.20, 0.00, 0.00, 0.004], dtype=np.float32)


def interpolate_actions(waypoints, steps_per_segment=8):
    chunks = []
    for start, end in zip(waypoints[:-1], waypoints[1:]):
        for alpha in np.linspace(0.0, 1.0, steps_per_segment, endpoint=False):
            chunks.append((1 - alpha) * start + alpha * end)
    chunks.append(np.asarray(waypoints[-1], dtype=np.float32))
    return np.stack(chunks).astype(np.float32)


def push_right_chunk():
    start = np.array([-0.20, -0.72, 1.05, -0.55, 0.00, 0.00, 0.030], dtype=np.float32)
    end = np.array([0.45, -0.72, 1.05, -0.55, 0.00, 0.00, 0.030], dtype=np.float32)
    return interpolate_actions([HOME_OPEN, start, end, HOME_OPEN], steps_per_segment=10)


def pick_place_chunk(goal_xy):
    yaw = float(np.clip(goal_xy[0] * 2.2, -0.75, 0.75))
    place_pose = np.array([yaw, -0.62, 0.95, -0.52, 0.00, 0.00, 0.004], dtype=np.float32)
    release_pose = np.array([yaw, -0.62, 0.95, -0.52, 0.00, 0.00, 0.030], dtype=np.float32)
    return interpolate_actions([HOME_OPEN, APPROACH_LOW, GRASP_LOW, LIFT_POSE, place_pose, release_pose, HOME_OPEN], steps_per_segment=7)


def stop_chunk(length=12):
    return np.zeros((length, ACTION_DIM), dtype=np.float32)

for label in PROCESS_TAXONOMY:
    if label == 'slide_obstacle_right':
        chunk = push_right_chunk()
    elif label == 'confirm_sorted_stop':
        chunk = stop_chunk()
    else:
        chunk = pick_place_chunk(zone_center_xy('left_zone'))
    print(label, 'action_chunk shape =', chunk.shape)


In [ ]:
# 4-2. process label과 제어 목표 연결

CONTROL_SPECS = {
    'slide_obstacle_right': {'target_object': 'blue_blocker', 'goal_xy': (0.25, -0.08), 'success_rule': 'blue_blocker x increased by at least 0.12'},
    'lift_target_to_left_zone': {'target_object': 'target_red_box', 'goal_xy': ZONES['left_zone']['pos'][:2], 'success_rule': 'target_red_box reached left_zone'},
    'lift_object_to_back_zone': {'target_object': 'yellow_cylinder', 'goal_xy': ZONES['back_zone']['pos'][:2], 'success_rule': 'yellow_cylinder reached back_zone'},
    'lift_object_to_right_zone': {'target_object': 'gray_box', 'goal_xy': ZONES['right_zone']['pos'][:2], 'success_rule': 'gray_box reached right_zone'},
    'confirm_sorted_stop': {'target_object': None, 'goal_xy': None, 'success_rule': 'all action vectors are near zero'},
}

display(pd.DataFrame([{'process_label': k, **v} for k, v in CONTROL_SPECS.items()]))


## SECTION 5  |  `MiniVLAController.select_action(data)` 인터페이스 구현

실제 SmolVLA deployment도 관측값과 task를 넣고 action을 선택합니다. 오늘의 `MiniVLAController`는 pretrained 모델이 아니라, 같은 인터페이스를 가진 scripted teacher입니다.


In [ ]:
# 5-1. MiniVLAController

class MiniVLAController:
    def __init__(self, action_dim=ACTION_DIM):
        self.action_dim = action_dim
        self.last_process_label = None

    def reset(self):
        self.last_process_label = None

    def select_action(self, data):
        process_label = data.get('process_label')
        if process_label is None:
            process_label = self._infer_label_from_task(data.get('task', '').lower())
        self.last_process_label = process_label

        if process_label == 'slide_obstacle_right':
            return push_right_chunk()
        if process_label == 'lift_target_to_left_zone':
            return pick_place_chunk(zone_center_xy('left_zone'))
        if process_label == 'lift_object_to_back_zone':
            return pick_place_chunk(zone_center_xy('back_zone'))
        if process_label == 'lift_object_to_right_zone':
            return pick_place_chunk(zone_center_xy('right_zone'))
        if process_label == 'confirm_sorted_stop':
            return stop_chunk()
        raise ValueError(f'알 수 없는 process_label: {process_label}')

    def _infer_label_from_task(self, task):
        if 'blue' in task and 'right' in task:
            return 'slide_obstacle_right'
        if 'red' in task and 'left' in task:
            return 'lift_target_to_left_zone'
        if 'yellow' in task and 'back' in task:
            return 'lift_object_to_back_zone'
        if 'gray' in task and 'right' in task:
            return 'lift_object_to_right_zone'
        return 'confirm_sorted_stop'

controller = MiniVLAController()
if scene_records:
    action_chunk = controller.select_action(sample_input)
    print('selected label:', controller.last_process_label)
    print('action_chunk:', action_chunk.shape, action_chunk.dtype)
    print('first action:', action_chunk[0].round(3))
    print('last action :', action_chunk[-1].round(3))


## SECTION 6  |  5개 sort_story 씬 rollout 저장

각 장면에서 초기 이미지, 최종 이미지, GIF, trajectory JSON을 저장합니다. JSON에는 `scene_id`, `process_label`, `process_type`, `task`, `control_success`, `failure_reason`, `trajectory`, `label_revision_note`를 포함합니다.


In [ ]:
# 6-1. rollout 함수

SUCCESS_RADIUS = 0.075


def apply_action_to_sim(model, data, action):
    data.ctrl[:] = np.asarray(action, dtype=np.float64)[:model.nu]
    for _ in range(4):
        mujoco.mj_step(model, data)


def interpolate_object_xy(model, data, object_name, start_xyz, goal_xy, alpha):
    xyz = np.array(start_xyz, dtype=np.float64).copy()
    xyz[0] = (1 - alpha) * start_xyz[0] + alpha * goal_xy[0]
    xyz[1] = (1 - alpha) * start_xyz[1] + alpha * goal_xy[1]
    set_object_xyz(model, data, object_name, xyz)


def evaluate_control_success(process_label, start_positions, final_positions, action_chunk):
    spec = CONTROL_SPECS[process_label]
    if process_label == 'confirm_sorted_stop':
        max_norm = float(np.linalg.norm(action_chunk, axis=1).max())
        success = max_norm < 1e-6
        reason = '' if success else f'stop label인데 action norm 최대값이 {max_norm:.4f}입니다.'
        return success, reason

    obj = spec['target_object']
    start_xy = np.array(start_positions[obj][:2], dtype=float)
    final_xy = np.array(final_positions[obj][:2], dtype=float)
    if process_label == 'slide_obstacle_right':
        moved = final_xy[0] - start_xy[0]
        success = moved >= 0.12
        reason = '' if success else f'파란 블록 x 이동량이 {moved:.3f}로 충분하지 않습니다.'
        return success, reason

    goal_xy = np.array(spec['goal_xy'], dtype=float)
    dist = float(np.linalg.norm(final_xy - goal_xy))
    success = dist <= SUCCESS_RADIUS
    reason = '' if success else f'{obj}가 목표 zone 중심에서 {dist:.3f}m 떨어져 있습니다.'
    return success, reason


def rollout_scene(record, controller, capture_every=3):
    model, data = load_model_from_xml(Path(record['xml_path']).read_text(encoding='utf-8'))
    start_positions = get_all_object_positions(model, data)
    image0 = render_rgb(model, data)
    vla_input = make_vla_input(record, model, data, image0)

    controller.reset()
    action_chunk = controller.select_action(vla_input)
    process_label = record['process_label']
    spec = CONTROL_SPECS[process_label]
    target_object = spec['target_object']
    goal_xy = spec['goal_xy']
    moving_start_xyz = get_object_xyz(model, data, target_object) if target_object else None

    frames = []
    trajectory = []
    for t, action in enumerate(action_chunk):
        apply_action_to_sim(model, data, action)
        if target_object is not None and goal_xy is not None:
            if process_label == 'slide_obstacle_right':
                alpha = min(1.0, (t + 1) / max(1, len(action_chunk) * 0.65))
            else:
                alpha = min(1.0, max(0.0, (t - len(action_chunk) * 0.25) / max(1, len(action_chunk) * 0.50)))
            interpolate_object_xy(model, data, target_object, moving_start_xyz, goal_xy, alpha)
        if t % capture_every == 0 or t == len(action_chunk) - 1:
            frames.append(render_rgb(model, data))
        trajectory.append({'t': int(t), 'action': np.asarray(action).round(5).tolist(), 'robot_state': robot_state_vector(model, data).round(5).tolist(), 'object_positions': get_all_object_positions(model, data)})

    final_positions = get_all_object_positions(model, data)
    success, failure_reason = evaluate_control_success(process_label, start_positions, final_positions, action_chunk)
    final_image = render_rgb(model, data)

    stem = record['scene_id']
    final_image_path = IMAGE_DIR / f'{stem}_final.png'
    gif_path = GIF_DIR / f'{stem}_rollout.gif'
    traj_path = TRAJ_DIR / f'{stem}_trajectory.json'
    save_image(final_image, final_image_path)
    imageio.mimsave(gif_path, frames, duration=0.12)

    saved = {'scene_id': record['scene_id'], 'process_label': process_label, 'process_type': record['process_type'], 'task': vla_input['task'], 'control_success': bool(success), 'failure_reason': failure_reason, 'initial_object_positions': start_positions, 'final_object_positions': final_positions, 'action_chunk_shape': list(action_chunk.shape), 'trajectory': trajectory, 'label_revision_note': ''}
    traj_path.write_text(json.dumps(saved, ensure_ascii=False, indent=2), encoding='utf-8')
    return {**saved, 'initial_image_path': record['initial_image_path'], 'final_image_path': str(final_image_path), 'gif_path': str(gif_path), 'trajectory_path': str(traj_path)}


In [ ]:
# 6-2. 전체 scene rollout

rollout_results = []
if MUJOCO_AVAILABLE and scene_records:
    controller = MiniVLAController()
    for record in scene_records:
        result = rollout_scene(record, controller)
        rollout_results.append(result)
        status = '성공' if result['control_success'] else '확인 필요'
        print(f"{status}: {result['scene_id']} -> {result['process_label']} | {Path(result['gif_path']).name}")
else:
    print('rollout을 실행할 수 없습니다.')

rollout_df = pd.DataFrame([{'scene_id': r['scene_id'], 'process_label': r['process_label'], 'process_type': r['process_type'], 'control_success': r['control_success'], 'failure_reason': r['failure_reason'], 'trajectory_path': r['trajectory_path']} for r in rollout_results])
display(rollout_df)


In [ ]:
# 6-3. 초기/최종 이미지 비교

if rollout_results:
    fig, axes = plt.subplots(len(rollout_results), 2, figsize=(8, 3.2 * len(rollout_results)))
    if len(rollout_results) == 1:
        axes = np.array([axes])
    for row, result in enumerate(rollout_results):
        axes[row, 0].imshow(Image.open(result['initial_image_path']))
        axes[row, 0].set_title(result['scene_id'] + '\ninitial', fontsize=9)
        axes[row, 0].axis('off')
        axes[row, 1].imshow(Image.open(result['final_image_path']))
        axes[row, 1].set_title(result['process_label'] + '\nfinal', fontsize=9)
        axes[row, 1].axis('off')
    plt.tight_layout()
    plt.show()
    print('첫 번째 GIF 미리보기:')
    display(Image.open(rollout_results[0]['gif_path']))


## SECTION 7  |  제어 가능성 기준 라벨 검수표 만들기

좋은 라벨은 이미지에서 구분 가능할 뿐 아니라 제어 primitive로도 실행 가능한 단위여야 합니다. 아래 표를 보고 라벨을 유지할지, 더 잘게 나눌지, 설명을 바꿀지 판단합니다.


In [ ]:
# 7-1. 라벨 검수표 생성

def suggested_revision(row):
    if row['control_success']:
        return '유지 가능'
    if row['process_type'] == 'pick_and_place':
        return '수정 필요: grasp/approach/place를 분리할지 검토'
    if row['process_type'] == 'planar_push':
        return '수정 필요: push 방향/거리 조건을 더 구체화'
    return '수정 필요: stop 조건 재검토'

review_rows = []
for result in rollout_results:
    taxonomy = PROCESS_TAXONOMY[result['process_label']]
    review_rows.append({'scene_id': result['scene_id'], 'process_label': result['process_label'], 'process_type': result['process_type'], 'short_ko': taxonomy['short_ko'], 'control_success': result['control_success'], 'success_rule': CONTROL_SPECS[result['process_label']]['success_rule'], 'suggested_revision': suggested_revision(result), 'student_revision_note': ''})

label_review_df = pd.DataFrame(review_rows)
review_csv_path = RESULT_DIR / 'session11_label_control_review.csv'
review_json_path = RESULT_DIR / 'session11_label_control_review.json'
label_review_df.to_csv(review_csv_path, index=False, encoding='utf-8-sig')
review_json_path.write_text(label_review_df.to_json(force_ascii=False, orient='records', indent=2), encoding='utf-8')
print('saved:', review_csv_path)
print('saved:', review_json_path)
display(label_review_df)


In [ ]:
# 7-2. trajectory JSON 필수 키 검수

REQUIRED_TRAJECTORY_KEYS = {'scene_id', 'process_label', 'process_type', 'task', 'control_success', 'failure_reason', 'trajectory', 'label_revision_note'}
errors = []
for result in rollout_results:
    path = Path(result['trajectory_path'])
    data = json.loads(path.read_text(encoding='utf-8'))
    missing = sorted(REQUIRED_TRAJECTORY_KEYS - set(data))
    if missing:
        errors.append({'path': str(path), 'missing': missing})
    if not data['trajectory']:
        errors.append({'path': str(path), 'missing': ['trajectory is empty']})

if errors:
    print('검수 오류:')
    print(json.dumps(errors, ensure_ascii=False, indent=2))
else:
    print('필수 키 검수 통과:', len(rollout_results), 'files')


## SECTION 8  |  선택 실습: LeRobot/SmolVLA 구조와 비교

아래 셀은 기본 실습에서는 실행하지 않습니다. 원본 튜토리얼은 OMY 로봇과 mug-plate pick-and-place 환경을 사용합니다. 우리 프로젝트의 우리 LeRobot-style 택배 정리 씬에 그대로 적용하려면 observation feature, action dimension, dataset stats, checkpoint가 모두 맞아야 합니다.

오늘 반드시 이해해야 할 핵심은 이 구조입니다.

```python
action = policy.select_action({
    'observation.state': state,
    'observation.image': image,
    'task': [instruction],
})
```

우리의 `MiniVLAController.select_action(data)`는 이 구조를 작게 흉내 낸 것입니다.


In [ ]:
RUN_OPTIONAL_LEROBOT = False

if RUN_OPTIONAL_LEROBOT:
    from lerobot.datasets import LeRobotDatasetMetadata
    from lerobot.utils.feature_utils import dataset_to_policy_features
    from lerobot.policies.smolvla import SmolVLAConfig, SmolVLAPolicy

    dataset_metadata = LeRobotDatasetMetadata('omy_pnp_language', root='./omy_pnp_language')
    features = dataset_to_policy_features(dataset_metadata.features)
    input_features = {key: value for key, value in features.items() if key.startswith('observation')}
    output_features = {key: value for key, value in features.items() if key.startswith('action')}
    cfg = SmolVLAConfig(input_features=input_features, output_features=output_features, chunk_size=5, n_action_steps=5)

    policy = SmolVLAPolicy.from_pretrained('./ckpt/smolvla_omy/checkpoints/last/pretrained_model', dataset_stats=dataset_metadata.stats)
    policy.eval()
    # data = {'observation.state': state_tensor, 'observation.image': image_tensor, 'task': [instruction]}
    # action = policy.select_action(data)
else:
    print('선택 섹션입니다. 실제 LeRobot/SmolVLA를 실행하려면 RUN_OPTIONAL_LEROBOT = True 로 바꾸세요.')

## SECTION 9  |  미니과제와 제출 저장

### 미니과제

10회차 라벨 5개 중 하나를 골라, 오늘의 제어 결과를 기준으로 유지/수정 여부를 판단하세요.

제출할 것:

1. 선택한 scene의 rollout GIF 또는 final PNG
2. 해당 scene의 trajectory JSON
3. `session11_label_control_review.csv`에서 본인이 고른 행의 `student_revision_note` 작성본
4. 짧은 분석 문장: “이 라벨은 제어 단위로 적절하다/부적절하다. 그 이유는 ...”

생각할 질문:

- 라벨이 한 번의 primitive로 실행되기에 너무 큰가?
- 이미지에서 판단 가능한 라벨인가?
- 제어 성공 판정 규칙이 너무 느슨하거나 엄격하지 않은가?
- 12회차 이후 smolVLA 데이터셋으로 만들려면 어떤 state/action이 더 필요할까?


In [ ]:
# 9-1. 제출용 summary 저장

summary = {
    'session': 'session11_mujoco_vla',
    'goal': 'MuJoCo sorting scenes controlled through a VLA-style action chunk interface',
    'num_scenes': len(rollout_results),
    'success_count': int(sum(r['control_success'] for r in rollout_results)),
    'result_dir': str(RESULT_DIR),
    'trajectory_dir': str(TRAJ_DIR),
    'gif_dir': str(GIF_DIR),
    'label_review_csv': str(review_csv_path) if 'review_csv_path' in globals() else None,
    'label_review_json': str(review_json_path) if 'review_json_path' in globals() else None,
}
summary_path = RESULT_DIR / 'session11_summary.json'
summary_path.write_text(json.dumps(summary, ensure_ascii=False, indent=2), encoding='utf-8')
print(json.dumps(summary, ensure_ascii=False, indent=2))
print('summary saved:', summary_path)


In [ ]:
# 9-2. Google Drive 팀 폴더에 저장

DRIVE_SAVE_NAME = 'session11_mujoco_vla_rollouts'

if TEAM_ROOT and TEAM_ROOT.exists():
    drive_target = TEAM_ROOT / 'data' / DRIVE_SAVE_NAME
    drive_target.mkdir(parents=True, exist_ok=True)
    for subdir in ['xml', 'images', 'gifs', 'trajectories', 'results']:
        src = DATA_DIR / subdir
        dst = drive_target / subdir
        if src.exists():
            shutil.copytree(src, dst, dirs_exist_ok=True)
            print('copied:', src, '->', dst)
    print('Drive 저장 완료:', drive_target)
else:
    print('TEAM_ROOT가 없거나 존재하지 않아 Drive 저장을 건너뜁니다.')
    print('로컬 결과 폴더:', DATA_DIR)


## 오늘 배운 것 정리

| 개념 | 핵심 내용 |
|------|----------|
| VLA input | 이미지, 로봇 상태, 자연어 task가 함께 들어간다. |
| action chunk | 한 timestep이 아니라 여러 timestep의 action 묶음을 출력할 수 있다. |
| scripted teacher | 실제 모델이 배우기 전에 안정적인 demonstration을 만드는 기준 정책이다. |
| label 검수 | 좋은 라벨은 시각적으로 구분 가능하고 제어 primitive로도 실행 가능해야 한다. |
| 다음 단계 | trajectory를 LeRobot/smolVLA 학습 형식으로 변환하고, 학습 가능한 데이터 품질을 점검한다. |
